In [1]:
from opt_targeted_transfers import GapTargetedTransfers
from opt_targeted_transfers import Dataset, split
from data_loaders import load_data, PATH_TO_TRAIN_DATA, PATH_TO_TEST_DATA

In [2]:
# Make train and test set
train_data = load_data(PATH_TO_TRAIN_DATA)
test_data = load_data(PATH_TO_TEST_DATA)

train_dataset = Dataset(df=train_data, outcome='consumption_per_capita_per_day', weight='hh_wgt', covs=['hh_size', 'urban'])
train_dataset, validation_dataset = split(train_dataset)
test_covariate_dataset = Dataset(df=test_data, outcome=None, weight='hh_wgt', covs=['hh_size', 'urban'])
test_dataset = Dataset(df=test_data, outcome='consumption_per_capita_per_day', weight='hh_wgt', covs=['hh_size', 'urban'])

In [3]:
# Gap targeted transfers
tt = GapTargetedTransfers(c_bar=2.15, n_regressors=5)

In [4]:
# Fit quantile regressors
tt.fit(train_dataset=train_dataset, validation_dataset=validation_dataset)

Fitting quantile regressor for quantile 0.05


100%|██████████| 300/300 [00:03<00:00, 83.95it/s, val loss=0.024] 


Fitting quantile regressor for quantile 0.27499999999999997


100%|██████████| 300/300 [00:03<00:00, 81.91it/s, val loss=0.107]


Fitting quantile regressor for quantile 0.49999999999999994


100%|██████████| 300/300 [00:03<00:00, 87.59it/s, val loss=0.156]


Fitting quantile regressor for quantile 0.725


100%|██████████| 300/300 [00:03<00:00, 85.02it/s, val loss=0.169]


Fitting quantile regressor for quantile 0.95


100%|██████████| 300/300 [00:03<00:00, 85.27it/s, val loss=0.0827]


In [7]:
# Get optimal policy by solving the optimization problem.
tt.set_budget(0.01)
tt.run_opt(test_covariate_dataset)
# Evaluate policy. 
res = tt.evaluate(test_dataset)
res

{'initial_poverty_rate': 0.5774688939291257,
 'initial_poverty_gap': 0.44336819493851437,
 'post_transfer_poverty_gap': 0.44281626561918047,
 'post_transfer_poverty_rate': 0.5769551762446636,
 'policy_cost_per_capita': 0.0008535037839970796,
 'budget': 0.01,
 'policy_type': 'continuous_gap',
 'd': 2}

In [8]:
tt.set_budget(0.10)
tt.run_opt(test_covariate_dataset)
# Evaluate policy. 
res = tt.evaluate(test_dataset)
res

{'initial_poverty_rate': 0.5774688939291257,
 'initial_poverty_gap': 0.44336819493851437,
 'post_transfer_poverty_gap': 0.382440733581507,
 'post_transfer_poverty_rate': 0.5463696691176577,
 'policy_cost_per_capita': 0.09688775305926797,
 'budget': 0.1,
 'policy_type': 'continuous_gap',
 'd': 2}

In [7]:
tt.compute_auc(test_covariate_dataset=test_covariate_dataset, test_dataset=test_dataset, metrics=["post_transfer_poverty_rate",
                                                              "post_transfer_poverty_gap"], budgets=[0.05, 0.1, 0.5, 1.0, 2.0])

{'post_transfer_poverty_rate': {'auc': 0.42496606180818536,
  'results': [0.564071177761271,
   0.5463696691176577,
   0.3815870062541372,
   0.1549559386644251,
   0.0]},
 'post_transfer_poverty_gap': {'auc': 0.20691377555289897,
  'results': [0.41221988572083973,
   0.382440733581507,
   0.17562334263938573,
   0.04203814555508711,
   0.0]}}